# Projet Final MACSIN4A2225 - Data Analysis
**Membres de l'équipe :**
* Emma Coco
* Uliana Chernysheva
* Adrien Formoso

**Lien vers le dépôt GitHub :** [Insère ton lien GitHub public ici]

In [37]:
# Importation des bibliothèques nécessaires
import os
import pandas as pd
import numpy as np
import plotly.express as px
from dash import Dash, html, dcc, Input, Output
from sklearn.cluster import KMeans
from dash import no_update # <-- À rajouter tout en haut de la cellule avec tes imports si ce n'est pas fait !
import dash_bootstrap_components as dbc

### 1. Présentation du Dataset et Exploration (Tâche 1)
Ce jeu de données contient des mesures horaires de la qualité de l'air à Séoul. Il s'agit d'un jeu de données multidimensionnel comprenant une dimension temporelle (Date et heure), spatiale (Latitude, Longitude, Code de station), et analytique (Concentration de SO2, NO2, O3, CO, PM10 et PM2.5). L'objectif de cette étape est de charger les données et d'afficher leurs caractéristiques principales.

In [31]:
# Function to load and explore the dataset
# Input: file_path (string) - the path to the csv file
# Output: df (DataFrame) - the loaded dataframe
def load_and_explore_data(file_path):
    df = pd.read_csv(file_path)
    
    print("--- 1. Forme du dataset ---")
    print(f"Lignes : {df.shape[0]}, Colonnes : {df.shape[1]}\n")
    
    print("--- 2. Types de données ---")
    print(df.dtypes)
    
    print("\n--- 3. Valeurs manquantes par colonne ---")
    print(df.isnull().sum())
    
    print("\n--- 4. Plage de valeurs (Min/Max) pour les variables numériques ---")
    print(df.describe().loc[['min', 'max']])
    
    return df

In [43]:
def concat_historical_temp(origine_path):

    csv_path_list = [ origine_path +filename for filename in os.listdir(origine_path)]

    df_list = [pd.read_csv(path) for path in csv_path_list]

    pd.concat(df_list).to_parquet("./temperature.parquet")

    return pd.concat(df_list)

concat_historical_temp("./temp data/raw data/")

,name,datetime,tempmax,tempmin,temp,feelslikemax,feelslikemin,feelslike,dew,humidity,...,solarenergy,uvindex,severerisk,sunrise,sunset,moonphase,conditions,description,icon,stations
0,seoul,2016-01-01,42.0,23.1,33.5,40.1,22.5,31.4,25.8,74.9,...,6.5,3,NaN,2016-01-01T07:46:46,2016-01-01T17:23:38,0.71,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"47111099999,47098099999,47112099999,RKSM,47110..."
1,seoul,2016-01-02,51.0,33.3,42.3,51.0,27.2,39.4,34.6,74.4,...,6.9,4,NaN,2016-01-02T07:46:56,2016-01-02T17:24:25,0.75,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"47111099999,47098099999,47112099999,RKSM,47110..."
2,seoul,2016-01-03,50.1,35.4,42.6,50.1,35.4,41.8,37.8,84.0,...,4.1,2,NaN,2016-01-03T07:47:05,2016-01-03T17:25:14,0.78,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"47111099999,47098099999,47112099999,RKSM,47110..."
3,seoul,2016-01-04,42.0,30.5,37.9,40.3,24.7,33.1,21.5,54.8,...,10.0,5,NaN,2016-01-04T07:47:11,2016-01-04T17:26:04,0.81,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"47111099999,47098099999,47112099999,RKSM,47110..."
4,seoul,2016-01-05,34.0,24.0,28.6,27.9,18.3,23.2,5.4,38.4,...,9.3,5,NaN,2016-01-05T07:47:15,2016-01-05T17:26:55,0.84,Partially cloudy,Partly cloudy throughout the day.,partly-cloudy-day,"47111099999,47098099999,47112099999,RKSM,47110..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
726,seoul,2019-12-28,43.0,23.8,33.5,43.0,18.1,32.2,18.7,55.7,...,9.5,5,NaN,2019-12-28T07:45:47,2019-12-28T17:20:46,0.06,Partially cloudy,Becoming cloudy in the afternoon.,partly-cloudy-day,"47111099999,47112099999,47120099999,4711909999..."
727,seoul,2019-12-29,44.6,33.7,39.0,43.9,30.4,37.5,24.7,57.2,...,4.2,2,NaN,2019-12-29T07:46:05,2019-12-29T17:21:27,0.09,"Rain, Partially cloudy",Partly cloudy throughout the day with late aft...,rain,"47111099999,47120099999,47112099999,4711909999..."
728,seoul,2019-12-30,44.7,25.6,39.2,44.7,19.2,37.3,32.1,76.0,...,3.8,2,NaN,2019-12-30T07:46:21,2019-12-30T17:22:10,0.13,"Rain, Partially cloudy",Partly cloudy throughout the day with rain.,rain,"47111099999,47112099999,47120099999,4711909999..."
729,seoul,2019-12-31,23.0,13.8,19.1,22.4,3.2,12.7,1.4,46.3,...,10.9,5,NaN,2019-12-31T07:46:35,2019-12-31T17:22:54,0.16,Snow,Clear conditions throughout the day with late ...,snow,"47111099999,47098099999,47112099999,4712009999..."


### 1.5. Enrichissement des données (Point Bonus)
**Objectif :** Croiser les données de pollution avec des données météorologiques externes pour mieux interpréter les indicateurs (ex: influence de la température sur la concentration des gaz).

In [32]:
# Function to load and merge external weather data
# Input: df (DataFrame) - the main dataset, weather_path (string) - path to weather csv
# Output: df_merged (DataFrame) - the enriched dataset
def enrich_with_weather(df, weather_path):
    try:
        df_weather = pd.read_csv(weather_path)
        df['Measurement date'] = pd.to_datetime(df['Measurement date'])
        df_weather['Date'] = pd.to_datetime(df_weather['Date'])
        
        df_merged = pd.merge(df, df_weather, left_on='Measurement date', right_on='Date', how='left')
        print("\n[Succès] Dataset externe fusionné !")
        return df_merged
    except FileNotFoundError:
        print("\n[Info] Fichier météo introuvable. Création de données météo simulées pour la démonstration...")
        np.random.seed(42)
        df['Temperature'] = np.random.normal(15, 5, len(df))
        return df

### 2. Indicateur de Machine Learning : Clustering K-Means (Tâche 2 - Requête Avancée)
**Calcul :** Utilisation de l'algorithme K-Means de Scikit-Learn pour segmenter les données en 3 clusters basés sur la concentration combinée de tous les polluants.
**Interprétation :** Cet indicateur permet de créer une nouvelle variable "Niveau de Risque" (Faible, Modéré, Élevé) pour classifier la toxicité globale de l'air à un instant T, au lieu de regarder chaque gaz séparément.

In [33]:
# Function to apply K-Means clustering to classify pollution risk levels
# Input: df (DataFrame) - the main dataset
# Output: df_ml (DataFrame) - dataset with a new 'Risk_Cluster' column
def apply_kmeans_clustering(df):
    pollutants = ['SO2', 'NO2', 'O3', 'CO', 'PM10', 'PM2.5']
    df_ml = df.dropna(subset=pollutants).copy()
    
    # K-Means avec 3 clusters
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
    df_ml['Risk_Cluster'] = kmeans.fit_predict(df_ml[pollutants])
    
    # Mapping pour rendre les résultats lisibles par un humain
    df_ml['Risk_Cluster'] = df_ml['Risk_Cluster'].map({0: 'Risque Faible', 1: 'Risque Modéré', 2: 'Risque Élevé'})
    
    return df_ml

### 3. Indicateurs de Regroupement, Temporels et Spatiaux (Tâche 2 - Requêtes 1, 3 et 4)
* **Requête de regroupement :** Moyenne de tous les polluants groupée par le code de la station. *Interprétation : Identifie les stations les plus exposées globalement.*
* **Requête Temporelle :** Moyenne journalière des PM10. *Interprétation : Met en évidence les pics d'alerte aux particules fines selon les jours de l'année.*
* **Requête Spatiale :** Moyenne des polluants groupée par Latitude et Longitude. *Interprétation : Permet de cartographier la pollution exacte selon les coordonnées GPS des quartiers de Séoul.*

In [34]:
# Function to calculate the mean pollution per station
# Input: df (DataFrame) - the main dataset
# Output: df_grouped (DataFrame) - data aggregated by station code
def analyze_pollution_by_station(df):
    pollutants = ['SO2', 'NO2', 'O3', 'CO', 'PM10', 'PM2.5']
    df_grouped = df.groupby('Station code')[pollutants].mean().reset_index()
    return df_grouped

# Function to extract daily temporal trend for PM10
# Input: df (DataFrame) - the main dataset
# Output: df_time (DataFrame) - daily mean of PM10
def temporal_analysis(df):
    df['Measurement date'] = pd.to_datetime(df['Measurement date'])
    df_time = df.groupby(df['Measurement date'].dt.date)['PM10'].mean().reset_index()
    return df_time

# Function to extract spatio-temporal distribution for the animated map
# Input: df (DataFrame) - the main dataset
# Output: df_anim (DataFrame) - mean of pollutants by coordinates AND by day
def spatio_temporal_analysis(df):
    df['Date'] = pd.to_datetime(df['Measurement date']).dt.strftime('%Y-%m-%d')
    pollutants = ['PM10', 'PM2.5', 'NO2', 'SO2', 'O3', 'CO']
    
    # Agrégation par Date et Station
    df_anim = df.groupby(['Date', 'Latitude', 'Longitude', 'Station code'])[pollutants].mean().reset_index()
    df_anim = df_anim.sort_values('Date')
    
    # FIX 1 : Empêcher Plotly de planter à cause des valeurs négatives ou égales à zéro
    for p in pollutants:
        df_anim[p] = df_anim[p].apply(lambda x: max(x, 0.01))
        
    # # FIX 2 : Limiter l'animation aux 30 premiers jours (1 mois) pour ne pas faire exploser la mémoire du serveur
    # jours_uniques = df_anim['Date'].unique()
    # if len(jours_uniques) > 30:
    #     df_anim = df_anim[df_anim['Date'].isin(jours_uniques[:30])]
        
    return df_anim

### 4. Tableau de Bord Interactif (Tâche 3)
Le code ci-dessous génère un tableau de bord Dash. Il intègre un menu déroulant (Callback) permettant de modifier dynamiquement la carte spatiale pour afficher le polluant de son choix.

In [35]:


# Function to build and run the interactive dashboard
# Input: df_ml, df_grouped, df_time, df_spatial, df_corr (DataFrames)
# Output: None (Launches Dash server)
def run_interactive_dashboard(df_ml, df_grouped, df_time, df_spatial, df_corr):
    # Ajout d'un thème Bootstrap moderne (ex: FLATLY, LUX ou MINTY)
    app = Dash(__name__, external_stylesheets=[dbc.themes.FLATLY])
    pollutants = ['PM10', 'PM2.5', 'NO2', 'SO2', 'O3', 'CO']

    # --- Préparation des figures avec un design épuré (plotly_white) ---
    fig_corr = px.imshow(df_corr, text_auto=True, aspect="auto", 
                         title="Corrélations (Heatmap)", 
                         color_continuous_scale='RdBu_r')
    fig_corr.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=40, b=20))
    
    fig_box = px.box(df_ml, x='Risk_Cluster', y='PM2.5', color='Risk_Cluster',
                     title="Distribution PM2.5 / Risque")
    fig_box.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=40, b=20))
                     
    fig_pie = px.pie(df_ml, names='Risk_Cluster', title="Répartition des Risques",
                     color='Risk_Cluster', color_discrete_map={'Risque Faible':'#2ecc71', 'Risque Modéré':'#f1c40f', 'Risque Élevé':'#e74c3c'})
    fig_pie.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=40, b=20))

    fig_bar = px.bar(df_grouped, x='Station code', y='PM10', title="Moyenne des PM10 / Station", color='PM10')
    fig_bar.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=40, b=20))
    
    fig_time = px.line(df_time, x='Measurement date', y='PM10', title="Évolution journalière (PM10)")
    fig_time.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=40, b=20))

    # --- Layout Bootstrap (Grille et Cartes) ---
    app.layout = dbc.Container([
        
        # En-tête du Dashboard
        dbc.Row([
            dbc.Col(html.Div([
                html.H1("Dashboard Qualité de l'Air - Séoul", className="text-center text-primary mt-4 mb-2"),
                html.H5("Équipe : [Vos Noms] | Dataset : Seoul Air Quality", className="text-center text-muted mb-4")
            ]), width=12)
        ]),

        # Menu déroulant et Carte Spatiale dans une belle "Card"
        dbc.Row([
            dbc.Col(dbc.Card([
                dbc.CardBody([
                    html.Label("🌍 Sélectionnez le polluant à cartographier :", className="fw-bold mb-2"),
                    dcc.Dropdown(
                        id='pollutant-dropdown',
                        options=[{'label': p, 'value': p} for p in pollutants],
                        value='PM2.5',
                        clearable=False,
                        className="mb-3"
                    ),
                    dcc.Graph(id='interactive-map')
                ])
            ], className="shadow-sm mb-4"), width=12)
        ]),

        # Graphiques principaux (Barres et Lignes) côte à côte
        dbc.Row([
            dbc.Col(dbc.Card([dbc.CardBody(dcc.Graph(figure=fig_bar))], className="shadow-sm mb-4"), md=6),
            dbc.Col(dbc.Card([dbc.CardBody(dcc.Graph(figure=fig_time))], className="shadow-sm mb-4"), md=6)
        ]),

        # Graphiques analytiques (Heatmap, Boxplot, Pie) en 3 colonnes
        dbc.Row([
            dbc.Col(dbc.Card([dbc.CardBody(dcc.Graph(figure=fig_corr))], className="shadow-sm mb-4"), md=4),
            dbc.Col(dbc.Card([dbc.CardBody(dcc.Graph(figure=fig_box))], className="shadow-sm mb-4"), md=4),
            dbc.Col(dbc.Card([dbc.CardBody(dcc.Graph(figure=fig_pie))], className="shadow-sm mb-4"), md=4)
        ])

    ], fluid=True, style={"backgroundColor": "#f4f6f9", "padding": "20px"}) # Fond gris clair élégant

    # Callback inchangé pour l'animation
    @app.callback(
        Output('interactive-map', 'figure'),
        [Input('pollutant-dropdown', 'value')]
    )
    def update_map(selected_pollutant):
        try:
            fig = px.scatter_mapbox(df_spatial, lat="Latitude", lon="Longitude", 
                                    color=selected_pollutant, size=selected_pollutant, 
                                    hover_name="Station code", animation_frame="Date",
                                    size_max=80, opacity=0.8,
                                    title=f"Carte Spatio-Temporelle animée (1er mois) : {selected_pollutant}", 
                                    mapbox_style="carto-positron", zoom=10.5)
            fig.update_layout(margin={"r":0,"t":40,"l":0,"b":0})
            return fig
        except Exception as e:
            print(f"ERREUR DANS LE CALLBACK MAPBOX : {e}")
            from dash import no_update
            return no_update

    app.run(debug=True, use_reloader=False)

In [36]:
# Main block : Exécution de toutes les étapes du projet
if __name__ == "__main__":
    # Étape 1 : Collecte
    file_path = "data.csv" # /!\ Vérifie le nom /!\
    df = load_and_explore_data(file_path)
    df_clean = df.dropna().copy()
    
    # Étape 2 : Enrichissement
    weather_path = "seoul_weather.csv" # /!\ Vérifie le nom /!\
    # df_enriched = enrich_with_weather(df_clean, weather_path)
    
    # Étape 3 : Construction des indicateurs
    df_ml = apply_kmeans_clustering(df_clean) 
    df_grouped = analyze_pollution_by_station(df_ml) 
    df_time = temporal_analysis(df_ml) 
    
    # --- NOUVEAU : Analyse Spatio-Temporelle pour la carte animée ---
    df_spatial = spatio_temporal_analysis(df_ml) 
    
    # Calcul de la matrice de corrélation
    pollutants = ['SO2', 'NO2', 'O3', 'CO', 'PM10', 'PM2.5']
    df_corr = df_ml[pollutants].corr()
    
    # Étape 4 : Lancement du Dashboard Interactif Complet
    print("\nLancement du Super Dashboard avec Carte Animée...")
    print("Cliquez sur le lien local (ex: http://127.0.0.1:8050/) pour l'afficher.")
    run_interactive_dashboard(df_ml, df_grouped, df_time, df_spatial, df_corr)

--- 1. Forme du dataset ---
Lignes : 647511, Colonnes : 11

--- 2. Types de données ---
Measurement date        str
Station code          int64
Address                 str
Latitude            float64
Longitude           float64
SO2                 float64
NO2                 float64
O3                  float64
CO                  float64
PM10                float64
PM2.5               float64
dtype: object

--- 3. Valeurs manquantes par colonne ---
Measurement date    0
Station code        0
Address             0
Latitude            0
Longitude           0
SO2                 0
NO2                 0
O3                  0
CO                  0
PM10                0
PM2.5               0
dtype: int64

--- 4. Plage de valeurs (Min/Max) pour les variables numériques ---
     Station code   Latitude   Longitude    SO2     NO2    O3    CO    PM10  \
min         101.0  37.452357  126.835151 -1.000  -1.000  -1.0  -1.0    -1.0   
max         125.0  37.658774  127.136792  3.736  38.445  33.6  71

/var/folders/1y/zl83lr7s7wz_gkynk1vv4lrc0000gn/T/ipykernel_55119/1947341310.py:79: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(df_spatial, lat="Latitude", lon="Longitude",
/var/folders/1y/zl83lr7s7wz_gkynk1vv4lrc0000gn/T/ipykernel_55119/1947341310.py:79: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(df_spatial, lat="Latitude", lon="Longitude",
/var/folders/1y/zl83lr7s7wz_gkynk1vv4lrc0000gn/T/ipykernel_55119/1947341310.py:79: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(df_spatial, lat="Latitude", lon="Longitude",
/var/folders/1y/zl83lr7s7wz_gkynk1vv4lrc0000gn/T/ipykernel_55119/1947341310.py:79: DeprecationWarning: *scatter_mapbox